# Forward test: every step, in order

Mimics the forward test of one stored monthly list, one step per cell, so each number can be inspected. It is what `scripts/track_returns.py` does for every month at once, spelled out: take the list as it was published, buy every name on the list date at its adjusted close, and see what each group did afterwards. Nothing here changes a threshold or writes to the cache.

Open with `uv run --with jupyter --with yfinance jupyter lab notebooks/forward_test.ipynb`.

| Step | What it does |
|---|---|
| 1 | Load the month's stored list |
| 2 | Split it into top 50, other ranked and unranked |
| 3 | Download adjusted closes from the list date (SPY included) |
| 4 | Return of every name over each horizon |
| 5 | Each group against SPY |
| 6 | Pass versus fail for each criterion |
| 7 | Dividend cuts since the list date |
| 8 | The best and worst names, and a cross-check against the script |

## Settings
The only cell you edit.

In [ ]:
MONTH = "2026-10"          # the month under data/reports/ to test
LIST_DATE = ""             # the day the list was made, YYYY-MM-DD; "" = the date recorded for the month (snapshots.json or the defaults)
HORIZONS = (30, 91, 182)   # days after the list date; a horizon the data does not reach yet is skipped
TOP_N = 50                 # the recommended group

## Setup
Finds the repo root and imports the pure return functions from `src/div_invst/tracking.py`.

In [ ]:
import csv, json, re, statistics, sys
from datetime import date
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists() and (p / "scripts").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from div_invst.data import Store, yahoo
from div_invst.tracking import dividend_cut, group_stats, horizon_end, split_by_criterion, total_return

REPORTS = ROOT / "data" / "reports"
FOLDER = REPORTS / MONTH
assert re.fullmatch(r"\d{4}-\d{2}", MONTH) and FOLDER.is_dir(), f"no report folder {FOLDER}"
DEFAULT_DATES = {"2026-10": "2026-10-26", "2026-11": "2026-11-06"}   # as in scripts/track_returns.py
if not LIST_DATE:
    snaps = REPORTS / "snapshots.json"
    LIST_DATE = (json.loads(snaps.read_text()) if snaps.exists() else {}).get(MONTH) or DEFAULT_DATES.get(MONTH, "")
assert LIST_DATE, f"no list date for {MONTH}: set LIST_DATE"
TODAY = date.today().isoformat()
print(f"month {MONTH}   list date {LIST_DATE}   today {TODAY}")

## Step 1: load the stored list
The full `ranked.csv` when the month has one, else `top50.csv` (then there is no comparison group).

In [ ]:
path = FOLDER / "ranked.csv"
if not path.exists():
    path = FOLDER / "top50.csv"
    print("note: no ranked.csv for this month, so only the top 50 can be tested")
rows = list(csv.DictReader(open(path, encoding="utf-8")))
print(f"{path.name}: {len(rows)} rows, columns {', '.join(rows[0])[:120]}...")

## Step 2: the groups
Top N is the recommendation. Other ranked is the rest of the scored list. Unranked is below the coverage cutoff.

In [ ]:
groups = {
    f"top {TOP_N}": [r["ticker"] for r in rows if r["rank"] and int(r["rank"]) <= TOP_N],
    "other ranked": [r["ticker"] for r in rows if r["rank"] and int(r["rank"]) > TOP_N],
    "unranked": [r["ticker"] for r in rows if not r["rank"]],
}
for name, t in groups.items():
    print(f"{name:13} {len(t):5} tickers   e.g. {', '.join(t[:8])}")

## Step 3: prices
One batched download of adjusted closes (dividends and splits included, so a return is a total return) from the list date. Takes a minute or two for the full list; names Yahoo has no prices for come back empty and stay unpriced (never counted as 0%).

In [ ]:
tickers = sorted({t for g in groups.values() for t in g} | {"SPY"})
prices = yahoo.prices(tickers, LIST_DATE)
spy = prices["SPY"]
assert spy, "no SPY prices"
unpriced = [t for t in tickers if not prices.get(t)]
print(f"{len(tickers) - len(unpriced)} of {len(tickers)} priced; SPY {spy[0][0]} to {spy[-1][0]}; unpriced: {', '.join(unpriced[:15])}{' ...' if len(unpriced) > 15 else ''}")

## Step 4: return of every name
From the first close on or after the list date to the last close on or before the end of the horizon. "to date" ends on the latest close.

In [ ]:
windows = [(f"{d}d", horizon_end(LIST_DATE, d)) for d in HORIZONS if horizon_end(LIST_DATE, d) <= TODAY] + [("to date", TODAY)]
returns = {label: {t: total_return(prices.get(t, []), LIST_DATE, end) for t in tickers} for label, end in windows}
print("windows:", ", ".join(f"{l} (to {e})" for l, e in windows))
covered = [(l, e) for l, e in windows if total_return(spy, LIST_DATE, e) is not None]
print("covered by the data:", ", ".join(l for l, _ in covered) or "none yet (the list is too recent)")

## Step 5: each group against SPY
Equal-weight mean and median, the share of names that beat SPY (hit), and the mean excess return.

In [ ]:
pct = lambda v, sign=True: "-" if v is None else (f"{v:+.1%}" if sign else f"{v:.0%}")
print(f"{'window':8} {'group':13} {'n':>5} {'priced':>6} {'mean':>8} {'median':>8} {'hit':>5} {'excess':>8}")
for label, end in covered:
    spy_r = returns[label]["SPY"]
    for name, t in groups.items():
        s = group_stats({x: returns[label][x] for x in t}, spy_r)
        print(f"{label:8} {name:13} {s['n']:5} {s['priced']:6} {pct(s['mean']):>8} {pct(s['median']):>8} {pct(s['hit'], False):>5} {pct(s['excess']):>8}")
    print(f"{label:8} {'SPY':13} {'':>5} {'':>6} {pct(spy_r):>8}\n")

## Step 6: pass versus fail for each criterion
Mean return of the names that passed a criterion against those that failed it, to date, over every listed name that has a return. Unknown and n/a are left out. A few weeks of data say little: this is to see the mechanics.

In [ ]:
crits = sorted((c for c in rows[0] if re.fullmatch(r"c\d+", c)), key=lambda c: int(c[1:]))
label = covered[-1][0] if covered else None
if label:
    split = split_by_criterion(rows, returns[label], crits)
    print(f"window: {label}\ncriterion        pass (n, mean)       fail (n, mean)")
    for c, cell in split.items():
        print(f"{c:9} {cell['pass'][0]:8} {pct(cell['pass'][1]):>9}   {cell['fail'][0]:8} {pct(cell['fail'][1]):>9}")
else:
    print("no window is covered yet")

## Step 7: dividend cuts since the list date
From the cached dividend history (read only): a payment below 90% of the one before it. The cache knows payments only up to its last refresh.

In [ ]:
store = Store()
cuts = {}
for name, t in groups.items():
    hits = []
    for x in t:
        row = store.latest("yahoo", "yf_dividends", x)
        if row and dividend_cut(row[1], LIST_DATE):
            hits.append(x)
    cuts[name] = hits
    print(f"{name:13} {len(hits):4} of {len(t)}   {', '.join(hits[:12])}")
store.close()

## Step 8: best and worst, and a cross-check
The top group's best and worst names. The notebook uses the same `total_return` and `group_stats` as `scripts/track_returns.py`, so the step 5 table should match this month's section of `data/reports/tracking.md` once the script has run on the same day.

In [ ]:
if label:
    top = {x: returns[label][x] for x in groups[f"top {TOP_N}"] if returns[label][x] is not None}
    ordered = sorted(top.items(), key=lambda kv: kv[1])
    print(f"top {TOP_N}, {label}: worst 5 {[(t, pct(r)) for t, r in ordered[:5]]}")
    print(f"top {TOP_N}, {label}: best 5  {[(t, pct(r)) for t, r in ordered[-5:][::-1]]}")
    md_path = REPORTS / "tracking.md"
    print("\ncross-check file:", md_path if md_path.exists() else "not written yet (run scripts/track_returns.py)")